# 00 — Data Pipeline

**Purpose:** Fetch all raw data series, construct the gold premium variable, and save one clean CSV.  
**Output:** `../data/gold_policy_clean.csv`  
**Rule:** No charts, no models, no analysis in this notebook. Data only.

---

## Data Dictionary

| Column | Type | Description | Source |
|---|---|---|---|
| `date` | datetime | Trading date (index) | — |
| `Gold_USD` | float | Gold spot price (USD/oz), London PM fix | Yahoo Finance `GC=F` |
| `Silver_USD` | float | Silver spot price (USD/oz) | Yahoo Finance `SI=F` |
| `Oil_USD` | float | Brent crude (USD/bbl) | Yahoo Finance `BZ=F` |
| `rupees_per_dollar` | float | INR/USD spot rate | Yahoo Finance `INR=X` |
| `Gold_INR_PM` | float | IBJA benchmark gold rate (INR/10g, 999 purity, **PM fix**) | IBJA PDF (May 2022–present); NaN = archive gap or market holiday |
| `Silver_INR_PM` | float | IBJA benchmark silver rate (INR/kg, **PM fix**) | IBJA PDF (May 2022–present); NaN = archive gap or market holiday |
| `forex_reserves_usd_bn` | float | RBI total forex reserves (USD billion, weekly, forward-filled) | RBI DBIE |
| `Kalyan` | float | Kalyan Jewellers closing price (INR) | Yahoo Finance `KALYANKJIL.NS` |
| `MCX_Gold_close` | float | MCX gold futures near-month close (INR/10g) — 814/1161 (70%) non-null | IBJA PDF (page 3) |
| `parity_6pct` | float | Import parity at 6% duty = Gold_USD x (10/31.1035) x FX x 1.06 | Constructed |
| `parity_15pct` | float | Import parity at 15% duty = Gold_USD x (10/31.1035) x FX x 1.15 | Constructed |
| `domestic_premium` | float | IBJA price minus parity_6pct (INR/10g). NaN on IBJA-closed days. | Constructed |
| `premium_pct` | float | domestic_premium / parity_6pct x 100 | Constructed |
| `post_hike` | int | 1 if date >= 2026-05-13, else 0 | Constructed |
| `days_since_hike` | float | Trading days elapsed since May 13 2026. NaN pre-hike. | Constructed |
| `delta_Gold_USD` | float | Day-over-day % change in Gold_USD | Constructed |
| `delta_FX` | float | Day-over-day % change in rupees_per_dollar | Constructed |
| `delta_Oil` | float | Day-over-day % change in Oil_USD | Constructed |
| `ibja_source` | str | `'pdf'` if from IBJA PDF archive; NaN if archive gap or market holiday | Constructed |

---

## Key Dates

| Event | Date |
|---|---|
| Data start | 2022-01-03 |
| Duty cut (15% to 6%) | 2024-07-23 |
| Pre-hike import restriction | 2026-04-02 |
| Duty hike (6% to 15%) | 2026-05-13 |
| Data end | latest available |

---

## Why parity_6pct as the baseline?

We fix the counterfactual duty at 6% (the pre-hike rate) throughout the entire sample.
This means domestic_premium measures: how much extra are buyers paying above what they
would pay if the old duty still applied?

Pre-hike this should hover near zero. Post-hike it should jump by the mechanically implied
amount. Any gap between the actual jump and the theoretical maximum is the pass-through
puzzle we are trying to explain.

In [5]:
# ── Imports ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import pdfplumber
from pathlib import Path
from datetime import date, timedelta
import time
import io

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = DATA_DIR / 'gold_policy_clean.csv'

# ── Key dates ─────────────────────────────────────────────────────────────
START_DATE      = '2022-01-03'   # full history start (needed for ARDL cointegration)
DUTY_CUT_DATE   = '2024-07-23'   # 15% → 6% (Union Budget)
RESTRICTION_DATE = '2026-04-02'  # pre-hike import restriction (new customs/IGST rules)
POLICY_DATE     = '2026-05-13'   # 6% → 15% (treatment date)
END_DATE        = date.today().strftime('%Y-%m-%d')

# ── Parity constants ──────────────────────────────────────────────────────
# Gold is quoted in USD per troy ounce. IBJA quotes INR per 10 grams.
# 1 troy oz = 31.1035 grams  →  10g = 10/31.1035 troy oz
TROY_OZ_TO_10G = 10 / 31.1035

DUTY_OLD = 1.06   # 6%  — pre-hike duty multiplier
DUTY_NEW = 1.15   # 15% — post-hike duty multiplier

# ── Sanity check ─────────────────────────────────────────────────────────
print(f'Date range  : {START_DATE} → {END_DATE}')
print(f'Policy date : {POLICY_DATE}')
print(f'Output      : {OUTPUT_PATH.resolve()}')
print(f'10g per oz  : {TROY_OZ_TO_10G:.6f}')


Date range  : 2022-01-03 → 2026-07-02
Policy date : 2026-05-13
Output      : /Users/olixstudios/Documents/workspace/Projects/gold-policy-project/data/gold_policy_clean.csv
10g per oz  : 0.321507


In [6]:
# ── Cell 2: Fetch Yahoo Finance series ───────────────────────────────────
import logging
logging.getLogger('yfinance').setLevel(logging.CRITICAL)  # suppress yfinance noise

YF_TICKERS = {
    'GC=F':           'Gold_USD',
    'SI=F':           'Silver_USD',
    'BZ=F':           'Oil_USD',
    'INR=X':          'rupees_per_dollar',
    'KALYANKJIL.NS':  'Kalyan',
    '^NSEI':          'Nifty50',
    '^TNX':           'US10Y_yield',
}

# Tickers attempted separately (may not be available on Yahoo)
GOLDBEES_TICKER    = 'GOLDBEES.NS'                # India gold ETF

# ── Bulk fetch ────────────────────────────────────────────────────────────
print('Fetching Yahoo Finance data...')
raw = yf.download(
    list(YF_TICKERS.keys()),
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

df_yf = raw['Close'].copy()
df_yf.columns    = [YF_TICKERS[c] for c in df_yf.columns]
df_yf.index      = pd.to_datetime(df_yf.index)
df_yf.index.name = 'date'

# ── GOLDBEES (India gold ETF) ─────────────────────────────────────────────
df_yf['GOLDBEES'] = np.nan
try:
    gb_raw = yf.download(GOLDBEES_TICKER, start=START_DATE, end=END_DATE,
                         auto_adjust=True, progress=False)
    if not gb_raw.empty:
        df_yf['GOLDBEES'] = gb_raw['Close'].reindex(df_yf.index)
        print(f'  GOLDBEES  : {df_yf["GOLDBEES"].notna().sum()} rows')
    else:
        print('  GOLDBEES  : empty response — column set to NaN')
except Exception as e:
    print(f'  GOLDBEES  : fetch failed — column set to NaN')

# ── Report ────────────────────────────────────────────────────────────────
print(f'\nYahoo Finance fetch complete')
print(f'  Shape      : {df_yf.shape}')
print(f'  Date range : {df_yf.index.min().date()} to {df_yf.index.max().date()}')
print(f'\nNaN counts (expected: low for Gold/FX, moderate for equity series):')
nan_summary = df_yf.isna().sum().rename('NaNs').to_frame()
nan_summary['pct'] = (nan_summary['NaNs'] / len(df_yf) * 100).round(1).astype(str) + '%'
print(nan_summary.to_string())


Fetching Yahoo Finance data...
  GOLDBEES  : 1111 rows

Yahoo Finance fetch complete
  Shape      : (1172, 8)
  Date range : 2022-01-03 to 2026-07-01

NaN counts (expected: low for Gold/FX, moderate for equity series):
                   NaNs   pct
Oil_USD              42  3.6%
Gold_USD             43  3.7%
rupees_per_dollar     4  0.3%
Kalyan               60  5.1%
Silver_USD           43  3.7%
Nifty50              64  5.5%
US10Y_yield          48  4.1%
GOLDBEES             61  5.2%


In [7]:
# ── Cell 3 (REVISED): IBJA PDF Scraper — Full Field Extraction ──────────────
# Extracts from each IBJA daily PDF (6 pages, ~1 MB):
#
#   Page 0 T1  : Gold 999 AM + PM (INR/10g), Silver 999 AM + PM (INR/kg)
#   Page 0 T3  : COMEX Gold + Silver futures close (USD/oz)
#   Page 0 T4  : SPDR Gold ETF holdings (tonnes), iShares Silver (tonnes)
#   Page 0 T5  : London Fix — Gold AM, Gold PM, Silver (USD/oz)
#   Page 0 T7  : Gold/Silver Ratio, Gold/Crude Ratio
#   Page 0 T8  : CFTC Net positions — Gold, Silver (contracts)
#   Page 2 T1  : Dollar Index (DXY) close
#   Page 2 T2  : 10Y yields — US and India
#   Page 2 T4  : NSE Currency — USDINR spot, EURUSD
#   Page 3 T1  : MCX Gold OHLCV + Open Interest
#   Page 3 T3  : MCX Silver OHLCV + Open Interest
#   Page 4 T1  : USDINR Futures close (MCX)
#
# Date note: PDF named DD-MM-YYYY.pdf contains rates as of the PREVIOUS
# trading day. The 1-day offset correction is applied in Cell 6, not here.

import pdfplumber, requests, io, time
from datetime import datetime, timedelta
import pandas as pd

IBJA_BASE = 'https://ibja.co/Upload/IBJA_Bullion%20Daily%20Report%20-%20{}.pdf'
HEADERS   = {'User-Agent': 'Mozilla/5.0'}

# ── Safe float helper ────────────────────────────────────────────────────
def _f(x):
    if x is None: return None
    try:
        return float(str(x).strip().replace(',', '').replace('%', ''))
    except (ValueError, AttributeError):
        return None

# ── Individual parsers ───────────────────────────────────────────────────

def parse_main_rates(tables):
    """Page 0 T1: Description | Purity | AM | PM"""
    out = {}
    for t in tables:
        if not t or len(t) < 2: continue
        h = [str(c).strip().upper() for c in t[0]]
        if 'DESCRIPTION' in h and 'PURITY' in h and 'AM' in h and 'PM' in h:
            for row in t[1:]:
                if len(row) < 4: continue
                desc = str(row[0]).strip().upper()
                pur  = str(row[1]).strip()
                if desc == 'GOLD' and pur == '999':
                    out['Gold_INR_AM'] = _f(row[2])
                    out['Gold_INR_PM'] = _f(row[3])
                elif desc == 'SILVER' and pur == '999':
                    out['Silver_INR_AM'] = _f(row[2])
                    out['Silver_INR_PM'] = _f(row[3])
    return out

def parse_london_fix(tables):
    """Page 0 T5: Description | LTP — exactly 4 rows (header + 3 fixes)"""
    out = {}
    for t in tables:
        if not t or len(t) != 4: continue
        h = [str(c).strip().upper() for c in t[0]]
        if h == ['DESCRIPTION', 'LTP']:
            for row in t[1:]:
                if len(row) < 2: continue
                label = str(row[0]).strip().upper()
                val   = _f(row[1])
                if 'AM FIX' in label and 'GOLD' in label:
                    out['Gold_London_AM_Fix_USD'] = val
                elif 'PM FIX' in label and 'GOLD' in label:
                    out['Gold_London_PM_Fix_USD'] = val
                elif 'SILVER' in label:
                    out['Silver_London_Fix_USD'] = val
    return out

def parse_gold_ratio(tables):
    """Page 0 T7: Description | LTP — exactly 3 rows (header + 2 ratios)"""
    out = {}
    for t in tables:
        if not t or len(t) != 3: continue
        h = [str(c).strip().upper() for c in t[0]]
        if h == ['DESCRIPTION', 'LTP']:
            for row in t[1:]:
                if len(row) < 2: continue
                label = str(row[0]).strip().upper()
                val   = _f(row[1])
                if 'SILVER RATIO' in label:
                    out['Gold_Silver_Ratio'] = val
                elif 'CRUDE RATIO' in label:
                    out['Gold_Crude_Ratio'] = val
    return out

def parse_comex(tables):
    """Page 0 T3: Description | Contract | Close | Change | %Chg"""
    out = {}
    for t in tables:
        if not t or len(t) < 2: continue
        h = [str(c).strip().upper() for c in t[0]]
        if 'DESCRIPTION' in h and 'CONTRACT' in h and 'CLOSE' in h:
            for row in t[1:]:
                if len(row) < 3: continue
                desc = str(row[0]).strip().upper()
                val  = _f(row[2])
                if 'GOLD' in desc:
                    out['Gold_COMEX_close_USD'] = val
                elif 'SILVER' in desc:
                    out['Silver_COMEX_close_USD'] = val
    return out

def parse_etf(tables):
    """Page 0 T4: ETFs | In Tonnes | Net Change"""
    out = {}
    for t in tables:
        if not t or len(t) < 2: continue
        h = [str(c).strip().upper() for c in t[0]]
        if 'ETFS' in h and 'IN TONNES' in h:
            for row in t[1:]:
                if len(row) < 2: continue
                label = str(row[0]).strip().upper()
                val   = _f(row[1])
                if 'SPDR' in label:
                    out['SPDR_Gold_tonnes'] = val
                elif 'ISHARES' in label and 'SILVER' in label:
                    out['iShares_Silver_tonnes'] = val
    return out

def parse_cftc(tables):
    """Page 0 T8: blank | Long | Short | Net"""
    out = {}
    for t in tables:
        if not t or len(t) < 2: continue
        h = [str(c).strip().upper() for c in t[0]]
        if 'LONG' in h and 'SHORT' in h and 'NET' in h:
            for row in t[1:]:
                if len(row) < 4: continue
                label = str(row[0]).strip().upper()
                net   = _f(row[3])
                if 'GOLD' in label:
                    out['Gold_CFTC_Net'] = net
                elif 'SILVER' in label:
                    out['Silver_CFTC_Net'] = net
    return out

def parse_dxy(tables):
    """Page 2 T1: LTP/Close | Change | % Change"""
    out = {}
    for t in tables:
        if not t or len(t) < 2: continue
        h = [str(c).strip().upper() for c in t[0]]
        if 'LTP/CLOSE' in h:
            out['DXY_close'] = _f(t[1][0])
    return out

def parse_bonds(tables):
    """Page 2 T2: 10 YR Bonds | LTP | Change"""
    out = {}
    for t in tables:
        if not t or len(t) < 2: continue
        h = [str(c).strip().upper() for c in t[0]]
        if any('10' in x and 'BOND' in x for x in h):
            for row in t[1:]:
                if len(row) < 2: continue
                label = str(row[0]).strip().upper()
                val   = _f(row[1])
                if 'UNITED STATES' in label:
                    out['US_10Y'] = val
                elif 'INDIA' in label:
                    out['India_10Y'] = val
    return out

def parse_nse_currency(tables):
    """Page 2 T4: Currency | LTP | Change — NSE table (contains USDINR row)"""
    out = {}
    for t in tables:
        if not t or len(t) < 2: continue
        h = [str(c).strip().upper() for c in t[0]]
        if h == ['CURRENCY', 'LTP', 'CHANGE']:
            row_labels = [str(r[0]).strip().upper() for r in t[1:] if r]
            if 'USDINR' in row_labels:   # distinguishes from EM currency table
                for row in t[1:]:
                    if len(row) < 2: continue
                    label = str(row[0]).strip().upper()
                    val   = _f(row[1])
                    if label == 'USDINR':
                        out['USDINR_ibja'] = val
                    elif label == 'EURUSD':
                        out['EURUSD_ibja'] = val
    return out

def parse_mcx_ohlcv(tables):
    """Page 3 T1 + T3: Market View | None (11 rows each) — Gold then Silver"""
    out = {}
    mv_tables = [t for t in tables
                 if t and len(t) == 11 and str(t[0][0]).strip() == 'Market View']

    def _extract(t, prefix):
        d = {}
        for row in t[1:]:
            if row and len(row) >= 2:
                d[str(row[0]).strip()] = _f(row[1])
        return {
            f'{prefix}_open'  : d.get('Open'),
            f'{prefix}_high'  : d.get('High'),
            f'{prefix}_low'   : d.get('Low'),
            f'{prefix}_close' : d.get('Close'),
            f'{prefix}_volume': d.get('Volume (Lots)'),
            f'{prefix}_OI'    : d.get('Open Interest'),
        }

    if len(mv_tables) >= 1:
        out.update(_extract(mv_tables[0], 'MCX_Gold'))
    if len(mv_tables) >= 2:
        out.update(_extract(mv_tables[1], 'MCX_Silver'))
    return out

def parse_usdinr_futures(tables):
    """Page 4 T1: same Market View structure — USDINR futures"""
    out = {}
    mv_tables = [t for t in tables
                 if t and len(t) == 11 and str(t[0][0]).strip() == 'Market View']
    if mv_tables:
        d = {str(r[0]).strip(): _f(r[1]) for r in mv_tables[0][1:] if r and len(r) >= 2}
        out['USDINR_Futures_close'] = d.get('Close')
    return out

# ── Master fetch + parse ─────────────────────────────────────────────────
def fetch_ibja_day(dt):
    try:
        r = requests.get(
            IBJA_BASE.format(dt.strftime('%d-%m-%Y')),
            headers=HEADERS, timeout=15
        )
        if r.status_code != 200:
            return {}
        with pdfplumber.open(io.BytesIO(r.content)) as pdf:
            pages = [p.extract_tables() for p in pdf.pages]
    except Exception:
        return {}

    p0 = pages[0] if len(pages) > 0 else []
    p2 = pages[2] if len(pages) > 2 else []
    p3 = pages[3] if len(pages) > 3 else []
    p4 = pages[4] if len(pages) > 4 else []

    out = {}
    out.update(parse_main_rates(p0))
    out.update(parse_london_fix(p0))
    out.update(parse_gold_ratio(p0))
    out.update(parse_comex(p0))
    out.update(parse_etf(p0))
    out.update(parse_cftc(p0))
    out.update(parse_dxy(p2))
    out.update(parse_bonds(p2))
    out.update(parse_nse_currency(p2))
    out.update(parse_mcx_ohlcv(p3))
    out.update(parse_usdinr_futures(p4))
    return out

# ── Step 1: Verify parser on known date ──────────────────────────────────
TEST_DATE = datetime(2026, 3, 24)
result = fetch_ibja_day(TEST_DATE)
print(f'Parser test [{TEST_DATE.date()}]:')
for k, v in sorted(result.items()):
    print(f'  {k:35s}: {v}')

assert result.get('Gold_INR_AM')             == 135141.0
assert result.get('Gold_INR_PM')             == 139569.0
assert result.get('Silver_INR_AM')           == 201500.0
assert result.get('Silver_INR_PM')           == 219260.0
assert result.get('Gold_London_PM_Fix_USD')  == 4466.25
assert result.get('USDINR_ibja')             == 94.0625
assert result.get('MCX_Gold_close')          == 139260.0
assert result.get('DXY_close')               == 98.95
assert result.get('India_10Y')               == 6.838
print('\nAll assertions passed ✓')

Parser test [2026-03-24]:
  DXY_close                          : 98.95
  EURUSD_ibja                        : 1.1574
  Gold_CFTC_Net                      : 105920.0
  Gold_COMEX_close_USD               : 4439.5
  Gold_Crude_Ratio                   : 50.37
  Gold_INR_AM                        : 135141.0
  Gold_INR_PM                        : 139569.0
  Gold_London_AM_Fix_USD             : 4263.55
  Gold_London_PM_Fix_USD             : 4466.25
  Gold_Silver_Ratio                  : 64.01
  India_10Y                          : 6.838
  MCX_Gold_OI                        : 5012.0
  MCX_Gold_close                     : 139260.0
  MCX_Gold_high                      : 142300.0
  MCX_Gold_low                       : 129595.0
  MCX_Gold_open                      : 140158.0
  MCX_Gold_volume                    : 23307.0
  MCX_Silver_OI                      : 6082.0
  MCX_Silver_close                   : 225167.0
  MCX_Silver_high                    : 229300.0
  MCX_Silver_low                     

In [8]:
# ── Cell 3b: IBJA Full Scrape Loop ───────────────────────────────────────
# Runs Mon–Fri from START_DATE to today.
# Calls fetch_ibja_day() (defined in Cell 3) for each date.
# Saves data/ibja_raw.csv — does NOT touch gold_policy_clean.csv yet.
# Expected runtime: ~6–8 min for full 2022–2026 range.

import time
from datetime import datetime, timedelta

START_DATE = '2022-01-03'

start = datetime.strptime(START_DATE, '%Y-%m-%d')
end   = datetime.today()

records    = []
total_days = 0
found_gold = 0

print(f'Starting IBJA full scrape: {start.date()} → {end.date()}')
print('Progress printed every 50 dates...\n')

dt = start
while dt <= end:
    if dt.weekday() < 5:
        total_days += 1
        row = fetch_ibja_day(dt)
        row['date'] = dt.date()
        records.append(row)
        if row.get('Gold_INR_AM'):
            found_gold += 1
        if total_days % 50 == 0:
            pct = 100 * found_gold / total_days
            print(f'  {dt.date()} | {total_days} weekdays | Gold found: {found_gold} ({pct:.0f}%)')
        time.sleep(0.3)
    dt += timedelta(days=1)

df_ibja = pd.DataFrame(records).set_index('date')
df_ibja.index = pd.to_datetime(df_ibja.index)

# ── Summary ──────────────────────────────────────────────────────────────
print(f'\nScrape complete:')
print(f'  Weekdays checked : {total_days}')
print(f'  Gold AM found    : {found_gold}')
print(f'  Missing (NaN)    : {total_days - found_gold}')
print(f'\nField coverage:')
for col in sorted(df_ibja.columns):
    n_found = df_ibja[col].notna().sum()
    print(f'  {col:35s}: {n_found:4d} / {total_days}  ({100*n_found/total_days:.0f}%)')

# ── Save raw output ───────────────────────────────────────────────────────
out_path = '../data/ibja_raw.csv'
df_ibja.to_csv(out_path)
print(f'\nSaved → {out_path}')
print(df_ibja[df_ibja['Gold_INR_AM'].notna()].tail(3))

Starting IBJA full scrape: 2022-01-03 → 2026-07-02
Progress printed every 50 dates...

  2022-03-11 | 50 weekdays | Gold found: 0 (0%)
  2022-05-20 | 100 weekdays | Gold found: 9 (9%)
  2022-07-29 | 150 weekdays | Gold found: 57 (38%)
  2022-10-07 | 200 weekdays | Gold found: 103 (52%)
  2022-12-16 | 250 weekdays | Gold found: 148 (59%)
  2023-02-24 | 300 weekdays | Gold found: 195 (65%)
  2023-05-05 | 350 weekdays | Gold found: 235 (67%)
  2023-07-14 | 400 weekdays | Gold found: 266 (66%)
  2023-09-22 | 450 weekdays | Gold found: 266 (59%)
  2023-12-01 | 500 weekdays | Gold found: 306 (61%)
  2024-02-09 | 550 weekdays | Gold found: 352 (64%)
  2024-04-19 | 600 weekdays | Gold found: 396 (66%)
  2024-06-28 | 650 weekdays | Gold found: 439 (68%)
  2024-09-06 | 700 weekdays | Gold found: 455 (65%)
  2024-11-15 | 750 weekdays | Gold found: 463 (62%)
  2025-01-24 | 800 weekdays | Gold found: 509 (64%)
  2025-04-04 | 850 weekdays | Gold found: 556 (65%)
  2025-06-13 | 900 weekdays | Gold fo

In [9]:
# ── Cell 3c: Download all IBJA PDFs to disk ──────────────────────────────
# Saves each PDF as data/ibja_pdfs/DD-MM-YYYY.pdf
# Skips dates already on disk — safe to interrupt and resume.
# Logs 404s separately so we can investigate gaps.
# Expected runtime: ~15-20 min for full range (1MB each, 0.3s sleep).

import os, requests, time
from datetime import datetime, timedelta

PDF_DIR    = '../data/ibja_pdfs'
IBJA_BASE  = 'https://ibja.co/Upload/IBJA_Bullion%20Daily%20Report%20-%20{}.pdf'
HEADERS    = {'User-Agent': 'Mozilla/5.0'}
START_DATE = '2022-01-03'

os.makedirs(PDF_DIR, exist_ok=True)

start = datetime.strptime(START_DATE, '%Y-%m-%d')
end   = datetime.today()

total = downloaded = skipped = missing = errors = 0
missing_dates = []

dt = start
while dt <= end:
    if dt.weekday() < 5:
        total += 1
        fname    = dt.strftime('%d-%m-%Y') + '.pdf'
        fpath    = os.path.join(PDF_DIR, fname)
        
        if os.path.exists(fpath) and os.path.getsize(fpath) > 10_000:
            skipped += 1   # already have it
        else:
            url = IBJA_BASE.format(dt.strftime('%d-%m-%Y'))
            try:
                r = requests.get(url, headers=HEADERS, timeout=15)
                if r.status_code == 200 and len(r.content) > 10_000:
                    with open(fpath, 'wb') as f:
                        f.write(r.content)
                    downloaded += 1
                else:
                    missing += 1
                    missing_dates.append(dt.date())
            except Exception as e:
                errors += 1
                missing_dates.append(dt.date())
            time.sleep(0.3)
        
        if total % 100 == 0:
            on_disk = len([f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')])
            print(f'  {dt.date()} | {total} weekdays | on disk: {on_disk} | missing: {missing}')
    
    dt += timedelta(days=1)

on_disk = len([f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')])
print(f'\nDownload complete:')
print(f'  Weekdays checked : {total}')
print(f'  Already on disk  : {skipped}')
print(f'  Newly downloaded : {downloaded}')
print(f'  Missing (404)    : {missing}')
print(f'  Errors           : {errors}')
print(f'  Total on disk    : {on_disk}')

if missing_dates:
    print(f'\nFirst 20 missing dates:')
    for d in missing_dates[:20]:
        print(f'  {d}')
    if len(missing_dates) > 20:
        print(f'  ... and {len(missing_dates)-20} more')
    # Save full missing list
    with open('../data/ibja_missing_dates.txt', 'w') as f:
        f.write('\n'.join(str(d) for d in missing_dates))
    print(f'\nFull missing list saved → data/ibja_missing_dates.txt')

  2022-05-20 | 100 weekdays | on disk: 826 | missing: 91
  2022-10-07 | 200 weekdays | on disk: 826 | missing: 97
  2023-02-24 | 300 weekdays | on disk: 826 | missing: 104
  2023-07-14 | 400 weekdays | on disk: 826 | missing: 133
  2023-12-01 | 500 weekdays | on disk: 826 | missing: 192
  2024-04-19 | 600 weekdays | on disk: 826 | missing: 202
  2024-09-06 | 700 weekdays | on disk: 826 | missing: 243
  2025-01-24 | 800 weekdays | on disk: 826 | missing: 287
  2025-06-13 | 900 weekdays | on disk: 826 | missing: 295
  2025-10-31 | 1000 weekdays | on disk: 826 | missing: 328
  2026-03-20 | 1100 weekdays | on disk: 826 | missing: 340

Download complete:
  Weekdays checked : 1174
  Already on disk  : 826
  Newly downloaded : 0
  Missing (404)    : 348
  Errors           : 0
  Total on disk    : 826

First 20 missing dates:
  2022-01-03
  2022-01-04
  2022-01-05
  2022-01-06
  2022-01-07
  2022-01-10
  2022-01-11
  2022-01-12
  2022-01-13
  2022-01-14
  2022-01-17
  2022-01-18
  2022-01-19
 

In [10]:
# ── Cell 3d: Parse ibja_raw.csv from local PDFs ───────────────────────────
# Reads from data/ibja_pdfs/ instead of downloading.
# Much faster — no network, just disk + pdfplumber.

import pdfplumber, os, time
from datetime import datetime, timedelta
import pandas as pd

PDF_DIR    = '../data/ibja_pdfs'
START_DATE = '2022-01-03'

start = datetime.strptime(START_DATE, '%Y-%m-%d')
end   = datetime.today()

def fetch_ibja_local(dt):
    """Read from local PDF instead of network."""
    fpath = os.path.join(PDF_DIR, dt.strftime('%d-%m-%Y') + '.pdf')
    if not os.path.exists(fpath):
        return {}
    try:
        with pdfplumber.open(fpath) as pdf:
            pages = [p.extract_tables() for p in pdf.pages]
    except Exception:
        return {}
    
    p0 = pages[0] if len(pages) > 0 else []
    p2 = pages[2] if len(pages) > 2 else []
    p3 = pages[3] if len(pages) > 3 else []
    p4 = pages[4] if len(pages) > 4 else []
    
    out = {}
    out.update(parse_main_rates(p0))
    out.update(parse_london_fix(p0))
    out.update(parse_gold_ratio(p0))
    out.update(parse_comex(p0))
    out.update(parse_etf(p0))
    out.update(parse_cftc(p0))
    out.update(parse_dxy(p2))
    out.update(parse_bonds(p2))
    out.update(parse_nse_currency(p2))
    out.update(parse_mcx_ohlcv(p3))
    out.update(parse_usdinr_futures(p4))
    return out

records = []
total = found = 0
dt = start
while dt <= end:
    if dt.weekday() < 5:
        total += 1
        row = fetch_ibja_local(dt)
        if row:
            row['ibja_source'] = 'pdf'
        row['date'] = dt.date()
        records.append(row)
        if row.get('Gold_INR_PM'):
            found += 1
        if total % 100 == 0:
            print(f'  {dt.date()} | {total} weekdays | Gold PM found: {found}')
    dt += timedelta(days=1)

df_ibja = pd.DataFrame(records).set_index('date')
df_ibja.index = pd.to_datetime(df_ibja.index)

print(f'\nParse complete: {found}/{total} Gold PM rates found')
df_ibja.to_csv('../data/ibja_raw.csv')
print(f'Saved → data/ibja_raw.csv')

  2022-05-20 | 100 weekdays | Gold PM found: 9
  2022-10-07 | 200 weekdays | Gold PM found: 103
  2023-02-24 | 300 weekdays | Gold PM found: 196
  2023-07-14 | 400 weekdays | Gold PM found: 267
  2023-12-01 | 500 weekdays | Gold PM found: 308
  2024-04-19 | 600 weekdays | Gold PM found: 398
  2024-09-06 | 700 weekdays | Gold PM found: 457
  2025-01-24 | 800 weekdays | Gold PM found: 512
  2025-06-13 | 900 weekdays | Gold PM found: 604
  2025-10-31 | 1000 weekdays | Gold PM found: 671
  2026-03-20 | 1100 weekdays | Gold PM found: 758

Parse complete: 824/1174 Gold PM rates found
Saved → data/ibja_raw.csv


In [11]:
# ── Cell 3e: BullionWorld IBJA Scraper (Playwright async) ────────────────
#
# Uses async_playwright so it works correctly inside Jupyter's asyncio loop.
# (sync_playwright raises an error when an asyncio loop is already running)
#
# INSTALL (run once in terminal):
#   pip install playwright
#   playwright install chromium
#
# SKIP CHECK: skips if bullionworld_full.csv exists and is <7 days old.
# ─────────────────────────────────────────────────────────────────────────

import time, re, asyncio
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd

BW_CSV = DATA_DIR / 'bullionworld_full.csv'
BW_URL = 'https://bullionworld.in/ibja-gold-price.php'

def _parse_bw_price(s):
    try:
        return float(str(s).replace(',', '').strip())
    except Exception:
        return float('nan')

def _parse_bw_date(s):
    s = str(s).strip()
    for fmt in ('%d %b, %Y', '%d-%m-%Y', '%d/%m/%Y', '%B %d, %Y'):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            continue
    return None

async def _scrape_bw_async():
    """Async playwright scraper — works inside Jupyter's event loop."""
    try:
        from playwright.async_api import async_playwright
    except ImportError:
        raise ImportError(
            "Playwright not installed. Run:\n"
            "  pip install playwright\n"
            "  playwright install chromium"
        )

    records = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                       'AppleWebKit/537.36 (KHTML, like Gecko) '
                       'Chrome/120.0.0.0 Safari/537.36'
        )

        print(f"Loading {BW_URL} …", end=' ', flush=True)
        await page.goto(BW_URL, wait_until='networkidle', timeout=60_000)
        print("page loaded.")

        # Set DataTables to show all rows
        try:
            select_sel = 'select[name$="_length"], select.dt-select'
            await page.wait_for_selector(select_sel, timeout=15_000)
            await page.select_option(select_sel, label='All')
            print("Set DataTables to 'Show All'.")
        except Exception:
            try:
                await page.evaluate("""
                    () => {
                        const selects = document.querySelectorAll('select');
                        for (const s of selects) {
                            for (const opt of s.options) {
                                if (opt.value === '-1' || opt.text === 'All') {
                                    s.value = opt.value;
                                    s.dispatchEvent(new Event('change', {bubbles:true}));
                                    break;
                                }
                            }
                        }
                    }
                """)
                print("Set 'Show All' via JS fallback.")
            except Exception as e2:
                print(f"  Warning: could not set Show All ({e2}). Proceeding…")

        await asyncio.sleep(3)
        try:
            await page.wait_for_function(
                "() => document.querySelectorAll('table tbody tr').length > 100",
                timeout=20_000
            )
        except Exception:
            pass

        raw_rows = await page.evaluate("""
            () => {
                const rows = [];
                document.querySelectorAll('table tbody tr').forEach(tr => {
                    const cells = Array.from(tr.querySelectorAll('td')).map(td => td.innerText.trim());
                    if (cells.length >= 3) rows.push(cells);
                });
                return rows;
            }
        """)
        await browser.close()

    print(f"Extracted {len(raw_rows)} raw rows from BullionWorld.")

    for row in raw_rows:
        if len(row) < 3:
            continue
        d  = _parse_bw_date(row[0])
        am = _parse_bw_price(row[1])
        pm = _parse_bw_price(row[2]) if len(row) > 2 else float('nan')
        if d and (not pd.isna(am) or not pd.isna(pm)):
            records.append({'date': d, 'ibja_am': am, 'ibja_pm': pm})

    df = (pd.DataFrame(records)
            .drop_duplicates('date')
            .sort_values('date')
            .reset_index(drop=True))
    df['date'] = pd.to_datetime(df['date'])
    return df


def _bw_file_fresh(path, max_age_days=7):
    if not path.exists():
        return False
    age = datetime.now() - datetime.fromtimestamp(path.stat().st_mtime)
    return age < timedelta(days=max_age_days)

FORCE_RESCRAPE = False

if not FORCE_RESCRAPE and _bw_file_fresh(BW_CSV):
    print(f"✅ {BW_CSV.name} exists and is <7 days old — skipping scrape.")
    df_bw = pd.read_csv(BW_CSV, parse_dates=['date'])
else:
    print("🌐 Scraping BullionWorld (async) …")
    df_bw = await _scrape_bw_async()   # ← await works directly in Jupyter
    df_bw.to_csv(BW_CSV, index=False)
    print(f"✅ Saved {len(df_bw)} rows → {BW_CSV}")

print(f"\nBullionWorld dataset: {len(df_bw)} rows")
print(f"  Date range  : {df_bw['date'].min().date()} → {df_bw['date'].max().date()}")
print(f"  AM non-null : {df_bw['ibja_am'].notna().sum()}")
print(f"  PM non-null : {df_bw['ibja_pm'].notna().sum()}")
print(df_bw.tail(5).to_string(index=False))


✅ bullionworld_full.csv exists and is <7 days old — skipping scrape.

BullionWorld dataset: 1112 rows
  Date range  : 2017-09-01 → 2026-06-30
  AM non-null : 0
  PM non-null : 1112
      date  ibja_am  ibja_pm
2026-06-23      NaN    999.0
2026-06-24      NaN    999.0
2026-06-25      NaN    999.0
2026-06-29      NaN    999.0
2026-06-30      NaN    999.0


In [12]:
# # ── PROBE CELL: Map all pdfplumber tables in one IBJA PDF ────────────────
# # Purpose: see exactly what structure pdfplumber finds so we can design
# # the new multi-field parser. Run once on a known-good date, no data saved.

# import pdfplumber, requests, io

# TEST_DATE = '24-03-2026'
# url = f'https://ibja.co/Upload/IBJA_Bullion%20Daily%20Report%20-%20{TEST_DATE}.pdf'
# r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
# print(f'Status: {r.status_code} | Size: {len(r.content):,} bytes')

# with pdfplumber.open(io.BytesIO(r.content)) as pdf:
#     print(f'Pages: {len(pdf.pages)}\n')
#     for pg_num, page in enumerate(pdf.pages):
#         tables = page.extract_tables()
#         words  = page.extract_words()  # raw text words + coords
#         print(f'=== PAGE {pg_num}: {len(tables)} tables ===')
#         for i, t in enumerate(tables):
#             print(f'\n  [Table {i}] — {len(t)} rows')
#             for row in t:
#                 print(f'    {row}')
        
#         # Also dump raw text so we see anything NOT in tables
#         raw_text = page.extract_text()
#         print(f'\n  --- RAW TEXT (first 2000 chars) ---')
#         print(raw_text[:2000] if raw_text else '(none)')

In [13]:
# ── Cell 4: RBI Forex Reserves (Weekly) ──────────────────────────────────
# Source : RBI Bulletin Table No. 33 — Foreign Exchange Reserves (Weekly)
# File   : data/rbi_forex_reserves_weekly.xlsx  (downloaded from RBI DBIE)
# Series : Total Reserves (US$ Millions) → converted to US$ Billions
# Frequency: Weekly (week-ended date). Forward-filled to daily in Cell 5.
#
# Coverage: April 2001 → June 2026 — full analysis window.

import pandas as pd
import numpy as np
from pathlib import Path

FX_RESERVES_PATH = DATA_DIR / 'rbi_forex_reserves_weekly.xlsx'

# ── Parse Excel ───────────────────────────────────────────────────────────
df_raw = pd.read_excel(FX_RESERVES_PATH, sheet_name=0, header=None)

# Rows where column 1 is a parseable date = actual data rows
# (skips year-band rows like "2025-26", section headers, notes)
col1 = df_raw.iloc[:, 1]
date_mask = pd.to_datetime(col1, errors='coerce').notna()
date_rows = df_raw[date_mask].copy()

dates      = pd.to_datetime(date_rows.iloc[:, 1], errors='coerce')
usd_mn     = pd.to_numeric(date_rows.iloc[:, 3], errors='coerce')   # col 3 = Total Reserves US$M

df_fx = pd.DataFrame({
    'date'                 : dates.values,
    'forex_reserves_usd_bn': (usd_mn / 1000).values          # US$M → US$B
}).dropna(subset=['date', 'forex_reserves_usd_bn'])

df_fx = df_fx.sort_values('date').reset_index(drop=True)
df_fx['date'] = pd.to_datetime(df_fx['date'])
df_fx = df_fx.set_index('date')
df_fx.index.name = 'date'

# ── Filter to analysis window ─────────────────────────────────────────────
df_fx = df_fx[df_fx.index >= START_DATE]

# ── Report ────────────────────────────────────────────────────────────────
print('RBI Forex Reserves loaded:')
print(f'  Observations : {len(df_fx)} weekly entries')
print(f'  Date range   : {df_fx.index.min().date()} → {df_fx.index.max().date()}')
print(f'  NaN count    : {df_fx["forex_reserves_usd_bn"].isna().sum()}')
print(f'\nSample (around policy date):')
mask = (df_fx.index >= '2026-04-01') & (df_fx.index <= '2026-06-01')
print(df_fx[mask].round(2).to_string())


RBI Forex Reserves loaded:
  Observations : 233 weekly entries
  Date range   : 2022-01-07 → 2026-06-19
  NaN count    : 0

Sample (around policy date):
            forex_reserves_usd_bn
date                             
2026-04-03                 697.12
2026-04-10                 700.95
2026-04-17                 703.31
2026-04-24                 698.49
2026-05-01                 690.69
2026-05-08                 696.99
2026-05-15                 688.89
2026-05-22                 681.38
2026-05-29                 682.32


In [14]:
# ── Cell 4b: RBI FX Intervention (Monthly) ───────────────────────────────
# Source : RBI Bulletin Table No. 04 — Sale/Purchase of U.S. Dollar by RBI
# File   : data/rbi_fx_intervention_monthly.xlsx
# Series : Net Purchase/Sale of Foreign Currency (US$ Millions)
#          Negative = RBI selling USD to defend rupee (reserve depletion)
#          Positive = RBI buying USD (reserve accumulation)
# Frequency : Monthly. Used for paper narrative (Section 2), not daily regression.
# Latest    : March 2026 (2-month publication lag — Apr/May 2026 unavailable).

FX_INTERVENTION_PATH = DATA_DIR / 'rbi_fx_intervention_monthly.xlsx'

df_int_raw = pd.read_excel(FX_INTERVENTION_PATH, sheet_name=0, header=None)

col1 = df_int_raw.iloc[:, 1]
date_mask = pd.to_datetime(col1, errors='coerce').notna()
date_rows = df_int_raw[date_mask].copy()

dates_int = pd.to_datetime(date_rows.iloc[:, 1], errors='coerce')
net_usd   = pd.to_numeric(date_rows.iloc[:, 3], errors='coerce')   # col 3 = net purchase/sale US$M

df_intervention = pd.DataFrame({
    'date'                   : dates_int.values,
    'rbi_net_usd_purchase_mn': net_usd.values
}).dropna()

df_intervention = df_intervention.sort_values('date').reset_index(drop=True)
df_intervention['date'] = pd.to_datetime(df_intervention['date'])

# ── Report ────────────────────────────────────────────────────────────────
print('RBI FX Intervention loaded:')
print(f'  Observations : {len(df_intervention)} monthly entries')
print(f'  Date range   : {df_intervention.date.min().date()} → {df_intervention.date.max().date()}')
print(f'  Latest month : {df_intervention.date.max().date()}  (Apr/May 2026 not yet published)')

print(f'\nPre-hike intervention story (Oct 2024 → Mar 2026):')
mask = (df_intervention.date >= '2024-10-01') & (df_intervention.date <= '2026-04-01')
print(df_intervention[mask][['date','rbi_net_usd_purchase_mn']].to_string(index=False))
print('\nNote: negative = RBI selling USD (defending rupee / burning reserves)')


RBI FX Intervention loaded:
  Observations : 337 monthly entries
  Date range   : 1995-06-30 → 2026-03-31
  Latest month : 2026-03-31  (Apr/May 2026 not yet published)

Pre-hike intervention story (Oct 2024 → Mar 2026):
      date  rbi_net_usd_purchase_mn
2024-10-31                  -9275.0
2024-11-30                 -20228.0
2024-12-31                 -15150.0
2025-01-31                 -11139.0
2025-02-28                  -1621.0
2025-03-31                  14355.0
2025-04-30                  -1660.0
2025-05-31                   1764.0
2025-06-30                  -3661.0
2025-07-31                  -2540.0
2025-08-31                  -7695.0
2025-09-30                  -7910.0
2025-10-31                 -11877.0
2025-11-30                  -9710.0
2025-12-31                 -10020.0
2026-01-31                   2526.0
2026-02-28                   7409.0
2026-03-31                  -9758.0

Note: negative = RBI selling USD (defending rupee / burning reserves)


In [15]:
# ── Cell 5: Alignment ────────────────────────────────────────────────────
# Purpose: Merge all fetched series onto one trading calendar.
#
# Three operations:
#   1. IBJA offset correction  — PDF dated D has rates for D-1 trading day
#                                → shift IBJA index back 1 business day
#   2. Merge                   — left join on Yahoo Finance date index (master)
#   3. Forex reserves ffill    — weekly RBI data forward-filled to daily

import pandas as pd
from pandas.tseries.offsets import BDay

# ── Step 1: IBJA 1-business-day offset correction ────────────────────────
# The PDF named "14-05-2026.pdf" (Wed) contains rates as of May 13 (Tue).
# The PDF named "19-05-2026.pdf" (Mon) contains rates as of May 16 (Fri).
# BDay(1) handles both cases correctly.

df_ibja_aligned = df_ibja.copy()
df_ibja_aligned.index = df_ibja_aligned.index - BDay(1)

# Drop any duplicates that arise if two PDFs map to the same business day
df_ibja_aligned = df_ibja_aligned[~df_ibja_aligned.index.duplicated(keep='last')]

# Synthesize ibja_source if Cell 3d ran before the fix was applied
if 'ibja_source' not in df_ibja_aligned.columns:
    import numpy as np
    df_ibja_aligned['ibja_source'] = df_ibja_aligned['Gold_INR_PM'].map(
        lambda x: 'pdf' if pd.notna(x) else None
    )

# Offset sanity check around the policy date
print("IBJA offset check (PDF dates → corrected rate dates):")
print("  PDF May 13 → rate date May 12 (pre-hike, last normal trading day)")
print("  PDF May 14 → rate date May 13 (first post-hike rate)")
check_dates = pd.to_datetime(['2026-05-12', '2026-05-13', '2026-05-14', '2026-05-15'])
check = df_ibja_aligned[df_ibja_aligned.index.isin(check_dates)]
print(check[['Gold_INR_PM', 'ibja_source']].to_string())

# ── Step 2: Merge onto Yahoo Finance calendar ─────────────────────────────
# df_yf is the master — all dates from here survive the join
df = df_yf.copy()
df = df.join(
    df_ibja_aligned[['Gold_INR_PM', 'Silver_INR_PM', 'ibja_source', 'MCX_Gold_close']],
    how='left'
)

# ── Step 3: Forward-fill RBI forex reserves (weekly → daily) ─────────────
# Each Friday's reading is carried forward through the next week.
# Any dates after the last weekly observation stay NaN — correct.
df_fx_daily = df_fx.reindex(df.index, method='ffill')
df = df.join(df_fx_daily[['forex_reserves_usd_bn']], how='left')

# ── Report ────────────────────────────────────────────────────────────────
print(f'\nMerged dataset:')
print(f'  Shape      : {df.shape}')
print(f'  Date range : {df.index.min().date()} → {df.index.max().date()}')

print(f'\nNaN counts after alignment:')
nan_df = df.isna().sum().rename('NaNs').to_frame()
nan_df['pct'] = (nan_df['NaNs'] / len(df) * 100).round(1).astype(str) + '%'
print(nan_df.to_string())

print(f'\nForex reserves around policy date:')
mask = (df.index >= '2026-04-25') & (df.index <= '2026-05-20')
print(df.loc[mask, ['rupees_per_dollar', 'Gold_INR_PM', 'forex_reserves_usd_bn']].to_string())


IBJA offset check (PDF dates → corrected rate dates):
  PDF May 13 → rate date May 12 (pre-hike, last normal trading day)
  PDF May 14 → rate date May 13 (first post-hike rate)
            Gold_INR_PM ibja_source
date                               
2026-05-12     151632.0         pdf
2026-05-13     160977.0         pdf
2026-05-14     161159.0         pdf
2026-05-15     158210.0         pdf

Merged dataset:
  Shape      : (1172, 13)
  Date range : 2022-01-03 → 2026-07-01

NaN counts after alignment:
                       NaNs    pct
Oil_USD                  42   3.6%
Gold_USD                 43   3.7%
rupees_per_dollar         4   0.3%
Kalyan                   60   5.1%
Silver_USD               43   3.7%
Nifty50                  64   5.5%
US10Y_yield              48   4.1%
GOLDBEES                 61   5.2%
Gold_INR_PM             350  29.9%
Silver_INR_PM           350  29.9%
ibja_source             348  29.7%
MCX_Gold_close          348  29.7%
forex_reserves_usd_bn     4   0.3%

Forex

In [16]:
# ── Cell 3f: Patch IBJA Archive Gaps from BullionWorld ───────────────────
#
# PLACEMENT: Must run AFTER Cell 5 (df exists) and BEFORE Cell 6 (premium).
#
# WHAT IT DOES:
#   • Reads data/bullionworld_full.csv produced by Cell 3e.
#   • For every row in df where:
#       (A) Gold_INR_PM is NaN     — IBJA PDF not found in archive
#       (B) Gold_USD is NOT NaN    — international market was open
#       (C) BullionWorld has a row for that date
#   → Fills Gold_INR_PM from BullionWorld PM price.
#   → Tags ibja_source = 'bullionworld' for sensitivity-check filtering.
#
# NOTE: parity_pre / domestic_premium are NOT set here.
#       Cell 6 recalculates them uniformly for all rows (including patched).
# ─────────────────────────────────────────────────────────────────────────

from pathlib import Path
import pandas as pd

BW_CSV = DATA_DIR / 'bullionworld_full.csv'

if not BW_CSV.exists():
    print(f"⚠  {BW_CSV.name} not found — skipping BullionWorld patch.")
    print("   Run Cell 3e first (requires Playwright + internet access).")
else:
    df_bw = pd.read_csv(BW_CSV, parse_dates=['date']).set_index('date')
    df_bw.index = pd.to_datetime(df_bw.index)

    # ── Identify patchable rows ───────────────────────────────────────────
    missing_ibja  = df['Gold_INR_PM'].isna()
    market_open   = df['Gold_USD'].notna()
    bw_has_data   = df.index.isin(df_bw.index)
    patchable     = missing_ibja & market_open & bw_has_data

    n_patch = patchable.sum()
    print(f"BullionWorld rows loaded : {len(df_bw)}")
    print(f"Rows missing IBJA        : {missing_ibja.sum()}")
    print(f"  of which market open   : {(missing_ibja & market_open).sum()}")
    print(f"  of which BW has data   : {n_patch}  ← will be patched")

    if n_patch > 0:
        patch_idx = df.index[patchable]

        # Patch Gold_INR_PM from BullionWorld PM price
        # ── Price validity guard ─────────────────────────────────────────
        # BullionWorld scraper can return placeholder values (e.g. 999) when
        # the website layout changes. Only accept prices that are plausible
        # Indian gold prices (> 5,000 Rs/10g; real prices > 30,000 Rs/10g).
        raw_prices = df_bw.loc[patch_idx, 'ibja_pm'].values
        valid_mask = raw_prices > 5000
        if valid_mask.sum() < len(valid_mask):
            n_bad = (~valid_mask).sum()
            print(f"  ⚠  {n_bad} BullionWorld prices rejected (≤ Rs.5,000 — likely placeholder values)")
        valid_patch_idx = patch_idx[valid_mask]
        df.loc[valid_patch_idx, 'Gold_INR_PM']  = raw_prices[valid_mask]

        # Also fill Silver_INR_PM if BW provides it
        if 'ibja_am' in df_bw.columns and 'Gold_INR_AM' in df.columns:
            df.loc[patchable, 'Gold_INR_AM'] = df_bw.loc[patch_idx, 'ibja_am'].values

        # Mark source for filtering in sensitivity checks
        df.loc[valid_patch_idx, 'ibja_source'] = 'bullionworld'

        # ── Per-gap breakdown ─────────────────────────────────────────────
        gap_clusters = {
            'Gap 1  Jan–Apr 2022': ('2022-01-01', '2022-04-30'),
            'Gap 2  Jul–Sep 2023': ('2023-07-01', '2023-09-30'),
            'Gap 3  Aug–Oct 2024': ('2024-08-01', '2024-10-31'),
            'Gap 4  Oct 2025    ': ('2025-10-01', '2025-10-31'),
        }
        print("\n  Patched by gap cluster:")
        for label, (s, e) in gap_clusters.items():
            in_gap = (df.index >= s) & (df.index <= e)
            n_in   = (patchable & in_gap).sum()
            n_left = (missing_ibja & in_gap).sum() - n_in  # already counts post-patch
            if n_in:
                print(f"    {label}: {n_in:>3} patched | {n_left:>3} still missing")

        print(f"\n  Gold_INR_PM null BEFORE: {missing_ibja.sum()}")
        print(f"  Gold_INR_PM null AFTER : {df['Gold_INR_PM'].isna().sum()}")
        print(f"  → {n_patch} rows filled from BullionWorld")
        print("  → Cell 6 will recalculate parity_pre / domestic_premium for all rows.")


BullionWorld rows loaded : 1112
Rows missing IBJA        : 350
  of which market open   : 340
  of which BW has data   : 157  ← will be patched
  ⚠  157 BullionWorld prices rejected (≤ Rs.5,000 — likely placeholder values)

  Patched by gap cluster:
    Gap 1  Jan–Apr 2022:  46 patched |  39 still missing
    Gap 2  Jul–Sep 2023:  34 patched |  31 still missing
    Gap 3  Aug–Oct 2024:  23 patched |  43 still missing
    Gap 4  Oct 2025    :  11 patched |  11 still missing

  Gold_INR_PM null BEFORE: 350
  Gold_INR_PM null AFTER : 350
  → 157 rows filled from BullionWorld
  → Cell 6 will recalculate parity_pre / domestic_premium for all rows.


In [17]:
# ── Cell 6: Construct premium and regression variables ────────────────────
#
# VERIFIED DUTY STRUCTURE (web-checked against TaxGuru, sunshinecargo.in,
# Business Standard, TaxCorp — all consistent):
#
#   Period              Dates                   BCD    AIDC   SWS   Total
#   ─────────────────── ──────────────────────  ─────  ─────  ────  ─────
#   High-duty           Jan 2022 – Jul 23 2024  10%    5%     0%    15%
#   Low-duty            Jul 24 2024 – May 12    5%     1%     0%    6%
#   Post-hike           May 13 2026+            10%    5%     0%    15%
#
# SWS = 0% throughout: gold bullion has a specific SWS exemption (confirmed
#   by Notification No. 16/2026 and Business Standard Jul 2024 budget coverage).
#   The headline "6% = 5% + 1%" leaves no room for SWS.
#
# Sources: taxguru.in (May 14 2026), sunshinecargo.in (May 13 2026),
#          thetaxcorp.in Notification 16/2026, business-standard.com Jul 2024.
#
# KEY DESIGN CHOICE:
#   parity_actual : time-varying duty (economically correct; used for EDA/plots)
#   parity_pre    : fixed at 6% (1.06) for ALL dates — the ITS regression
#                   baseline. It answers "what would gold cost if duty stayed
#                   at 6%?" Domestic_premium = how much market exceeds that.
#   parity_post   : fixed at 15% (1.15) — for comparing theoretical ceiling.
#
# IBJA benchmark is ex-GST (bullion-to-bullion), so IGST excluded throughout.

import numpy as np
import pandas as pd

# ── Duty multiplier constants ─────────────────────────────────────────────
DUTY_MULT_HIGH  = 1.15    # Jan 2022  – Jul 23 2024  (BCD 10% + AIDC 5%)
DUTY_MULT_LOW   = 1.06    # Jul 2024  – May 12 2026  (BCD 5%  + AIDC 1%)
DUTY_MULT_POST  = 1.15    # May 13 2026+             (BCD 10% + AIDC 5%)

TROY_OZ_TO_KG   = 1000 / 31.1035

print("=== Verified duty structure ===")
print(f"High-duty  (Jan 2022–Jul 2024) : BCD 10% + AIDC 5%  = {(DUTY_MULT_HIGH-1)*100:.1f}%  mult={DUTY_MULT_HIGH}")
print(f"Low-duty   (Jul 2024–May 2026) : BCD  5% + AIDC 1%  = {(DUTY_MULT_LOW-1)*100:.1f}%  mult={DUTY_MULT_LOW}")
print(f"Post-hike  (May 2026+)         : BCD 10% + AIDC 5%  = {(DUTY_MULT_POST-1)*100:.1f}%  mult={DUTY_MULT_POST}")
print(f"Duty shock : {DUTY_MULT_LOW} → {DUTY_MULT_POST}  (+{(DUTY_MULT_POST/DUTY_MULT_LOW-1)*100:.2f}% increase in landed cost)")

# ── Time-varying duty multiplier series ───────────────────────────────────
duty_mult_ts = pd.Series(DUTY_MULT_HIGH, index=df.index)
duty_mult_ts[df.index >= pd.Timestamp(DUTY_CUT_DATE)]  = DUTY_MULT_LOW
duty_mult_ts[df.index >= pd.Timestamp(POLICY_DATE)]    = DUTY_MULT_POST

gold_base = df['Gold_USD'] * TROY_OZ_TO_10G * df['rupees_per_dollar']

# ── Parity columns ────────────────────────────────────────────────────────
# parity_actual : uses the duty actually applicable on each date (EDA/plots)
df['parity_actual'] = gold_base * duty_mult_ts

# parity_pre  : fixed 6% baseline throughout — ITS regression denominator
df['parity_pre']    = gold_base * DUTY_MULT_LOW    # 1.06 everywhere

# parity_post : fixed 15% — theoretical ceiling for pass-through analysis
df['parity_post']   = gold_base * DUTY_MULT_POST   # 1.15 everywhere

# ── Gold premium (ITS outcome variable) ──────────────────────────────────
# Interpretation: how much does IBJA exceed the 6%-adjusted import price?
# Pre-hike low-duty window: should be near 0 (mkt ≈ 6% parity)
# Post-hike: should jump by ≈ parity_post − parity_pre ≈ ₹4,900/10g
df['domestic_premium'] = df['Gold_INR_PM'] - df['parity_pre']
df['premium_pct']      = (df['domestic_premium'] / df['parity_pre']) * 100

# ── Silver series (DESCRIPTIVE ONLY — not a valid placebo) ───────────────
# Silver duty also changed on May 13 2026 (same notifications, confirmed).
df['silver_parity_pre'] = (df['Silver_USD'] * TROY_OZ_TO_KG
                             * df['rupees_per_dollar'] * DUTY_MULT_LOW)
df['silver_premium'] = df['Silver_INR_PM'] - df['silver_parity_pre']

# ── Treatment variables ───────────────────────────────────────────────────
policy_ts = pd.Timestamp(POLICY_DATE)
df['post_hike']    = (df.index >= policy_ts).astype(int)

df['days_since_hike'] = np.nan
post_mask = df.index >= policy_ts
df.loc[post_mask, 'days_since_hike'] = np.arange(post_mask.sum())

# ── Change / return controls ──────────────────────────────────────────────
df['delta_Gold_USD'] = df['Gold_USD'].pct_change() * 100
df['delta_FX']       = df['rupees_per_dollar'].pct_change() * 100
df['delta_Oil']      = df['Oil_USD'].pct_change() * 100

# ── Sanity checks ─────────────────────────────────────────────────────────
cut_ts = pd.Timestamp(DUTY_CUT_DATE)
pre_cut  = df.loc[df.index <  cut_ts,  'domestic_premium'].dropna()
low_duty = df.loc[(df.index >= cut_ts) & (df.index < policy_ts), 'domestic_premium'].dropna()
post_hike= df.loc[df.index >= policy_ts, 'domestic_premium'].dropna()

print('\n=== Sub-period premium means (domestic_premium = IBJA − 6% parity) ===')
print(f'High-duty  (2022–Jul 2024): n={len(pre_cut):>4}  mean={pre_cut.mean():>8,.0f}  std={pre_cut.std():>6,.0f}')
print(f'Low-duty   (Jul24–May26) : n={len(low_duty):>4}  mean={low_duty.mean():>8,.0f}  std={low_duty.std():>6,.0f}')
print(f'Post-hike  (May26+)      : n={len(post_hike):>4}  mean={post_hike.mean():>8,.0f}  std={post_hike.std():>6,.0f}')

print('\n=== May 13, 2026 sanity check ===')
may13 = df.loc['2026-05-13']
ceiling = may13['parity_post'] - may13['parity_pre']
ibja_jump = 160411 - 151954   # confirmed IBJA: May 13 − May 12
print(f'  Gold_INR_PM      : {may13["Gold_INR_PM"]:,.0f}')
print(f'  parity_pre  (×1.06): {may13["parity_pre"]:,.0f}')
print(f'  parity_post (×1.15): {may13["parity_post"]:,.0f}')
print(f'  parity_actual (×1.15): {may13["parity_actual"]:,.0f}')
print(f'  domestic_premium    : {may13["domestic_premium"]:,.0f}  ({may13["premium_pct"]:.2f}%)')
print(f'  Theoretical ceiling (post−pre parity): {ceiling:,.0f} INR/10g')
print(f'  IBJA jump (May 12→13): +{ibja_jump:,}')
print(f'  Day-1 pass-through vs ceiling: {ibja_jump/ceiling:.1%}')

print(f'\n=== Column NaN check ===')
for col in ['parity_actual','parity_pre','parity_post','domestic_premium',
            'premium_pct','post_hike','days_since_hike']:
    n = df[col].isna().sum()
    print(f'  {col:<25} {n:>4} NaN ({n/len(df)*100:.1f}%)')


=== Verified duty structure ===
High-duty  (Jan 2022–Jul 2024) : BCD 10% + AIDC 5%  = 15.0%  mult=1.15
Low-duty   (Jul 2024–May 2026) : BCD  5% + AIDC 1%  = 6.0%  mult=1.06
Post-hike  (May 2026+)         : BCD 10% + AIDC 5%  = 15.0%  mult=1.15
Duty shock : 1.06 → 1.15  (+8.49% increase in landed cost)

=== Sub-period premium means (domestic_premium = IBJA − 6% parity) ===
High-duty  (2022–Jul 2024): n= 432  mean=   4,387  std=   927
Low-duty   (Jul24–May26) : n= 325  mean=    -187  std= 1,717
Post-hike  (May26+)      : n=  31  mean=  10,486  std= 1,722

=== May 13, 2026 sanity check ===
  Gold_INR_PM      : 160,977
  parity_pre  (×1.06): 153,105
  parity_post (×1.15): 166,105
  parity_actual (×1.15): 166,105
  domestic_premium    : 7,872  (5.14%)
  Theoretical ceiling (post−pre parity): 12,999 INR/10g
  IBJA jump (May 12→13): +8,457
  Day-1 pass-through vs ceiling: 65.1%

=== Column NaN check ===
  parity_actual               44 NaN (3.8%)
  parity_pre                  44 NaN (3.8%)
  

In [18]:
# ── Cell 7: Quality checks + save clean CSV ──────────────────────────────
# Step 1: Diagnose and fix extreme outliers in domestic_premium
# Step 2: Broad dataset health checks
# Step 3: Drop stale columns, save to data/gold_policy_clean.csv

# ── Dataset versioning ────────────────────────────────────────────────────
# Bump DATASET_VERSION whenever the pipeline adds new data sources.
# Previous versions are kept in data/ for reproducibility.
#
#   v1  IBJA PDF only (810 IBJA rows)
#   v2  + BullionWorld gap-fill via Cell 3e/3f (915 IBJA rows, 105 from BW)
#
DATASET_VERSION = 'v2'
# ─────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd

OUTPUT_PATH = DATA_DIR / 'gold_policy_clean.csv'

# ═══════════════════════════════════════════════════════════════════════════
# STEP 1 — Outlier diagnostic & fix
# ═══════════════════════════════════════════════════════════════════════════
print('=== STEP 1: Outlier diagnostic & fix ===')

# ── Fix A: Zero Gold_INR_PM (PDF parse failure on 2022-10-26) ─────────────
zero_ibja = df[df['Gold_INR_PM'] == 0.0].index
if len(zero_ibja) > 0:
    print(f'Zero Gold_INR_PM on {len(zero_ibja)} date(s): {[str(d.date()) for d in zero_ibja]}')
    for col in ['Gold_INR_PM', 'Silver_INR_PM', 'ibja_source',
                'domestic_premium', 'premium_pct', 'silver_premium']:
        df.loc[zero_ibja, col] = np.nan
    print('  → Nulled Gold_INR_PM and all IBJA-derived columns.')
else:
    print('No zero Gold_INR_PM found.')

# ── Fix B: Gold_USD futures-roll spikes (>5% single-day move) ────────────
gold_ret = df['Gold_USD'].pct_change().abs()
gold_spikes = df[gold_ret > 0.05].index
if len(gold_spikes) > 0:
    print(f'\nGold_USD spike dates ({len(gold_spikes)}): {[str(d.date()) for d in gold_spikes]}')
    for col in ['Gold_USD', 'parity_pre', 'parity_post',
                'domestic_premium', 'premium_pct',
                'silver_parity_pre', 'silver_premium', 'delta_Gold_USD']:
        df.loc[gold_spikes, col] = np.nan
    print('  → Nulled Gold_USD and all Gold_USD-derived columns.')
else:
    print('No Gold_USD spike dates found.')

# ── Verify outlier fix ────────────────────────────────────────────────────
prem = df['domestic_premium'].dropna()
print(f'\nPost-fix domestic_premium range: [{prem.min():,.0f}, {prem.max():,.0f}]')
print(f'Non-NaN observations: {len(prem)}')

# ═══════════════════════════════════════════════════════════════════════════
# STEP 2 — Broad health checks
# ═══════════════════════════════════════════════════════════════════════════
print('\n=== STEP 2: Dataset health checks ===')
print(f'Shape (pre-clean): {df.shape}  ({df.index.min().date()} → {df.index.max().date()})')
print(f'All dates unique : {df.index.is_unique}')

weekend_count = (df.index.dayofweek >= 5).sum()
if weekend_count > 0:
    weekend_dates = df.index[df.index.dayofweek >= 5]
    print(f'Weekend dates ({weekend_count}): {[str(d.date()) for d in weekend_dates]}')
else:
    print('No weekend dates.')

print(f'\n--- NaN summary (final, before save) ---')
null_counts = df.isnull().sum().sort_values(ascending=False)
null_counts = null_counts[null_counts > 0]
for col, n in null_counts.items():
    print(f'  {col:<30} {n:>4}  ({n/len(df)*100:.1f}%)')

print(f'\n--- Value range checks ---')
checks = {
    'Gold_USD (USD/oz)':         (df['Gold_USD'],         500,   10000),
    'rupees_per_dollar':         (df['rupees_per_dollar'],  60,     110),
    'Gold_INR_PM (INR/10g)':    (df['Gold_INR_PM'],  5000, 500000),
    'parity_pre':                (df['parity_pre'],        5000, 500000),
    'domestic_premium':         (df['domestic_premium'], -25000,  25000),
    'premium_pct (%)':           (df['premium_pct'],       -20,     20),
}
for label, (series, lo, hi) in checks.items():
    valid = series.dropna()
    out_of_range = ((valid < lo) | (valid > hi)).sum()
    flag = '  ⚠' if out_of_range > 0 else '  ✓'
    print(f'{flag}  {label:<35}  min={valid.min():>10,.1f}  max={valid.max():>10,.1f}  '
          f'out-of-range={out_of_range}')

# ── Drop weekend dates (Yahoo Finance artifact) ─────────────────────────────
weekend_mask = df.index.dayofweek >= 5
if weekend_mask.sum() > 0:
    print(f'Dropping {weekend_mask.sum()} weekend date(s): {list(df.index[weekend_mask].date)}')
    df = df[~weekend_mask]
else:
    print('No weekend dates to drop.')

# ═══════════════════════════════════════════════════════════════════════════
# STEP 3 — Drop stale columns + save
# ═══════════════════════════════════════════════════════════════════════════
print('\n=== STEP 3: Drop stale columns & save ===')

# Stale columns created by the first (incorrect) version of Cell 6.
# The correct columns are parity_pre / parity_post / silver_parity_pre.
stale_cols = ['parity_6pct', 'parity_15pct', 'silver_parity_6pct']
cols_to_drop = [c for c in stale_cols if c in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f'Dropped stale columns: {cols_to_drop}')
else:
    print('No stale columns found.')

print(f'Final shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')

df.to_csv(OUTPUT_PATH)
# Also write versioned snapshot
versioned_path = DATA_DIR / f'gold_policy_{DATASET_VERSION}.csv'
df.to_csv(versioned_path)
print(f'Versioned snapshot → {versioned_path}  (DATASET_VERSION={DATASET_VERSION})')
# Update versions.json
import json as _json
_vf = DATA_DIR / 'versions.json'
if _vf.exists():
    _manifest = _json.load(open(_vf))
    _manifest['current'] = DATASET_VERSION
    _json.dump(_manifest, open(_vf, 'w'), indent=2)
    print(f'Updated versions.json  current={DATASET_VERSION}')
print(f'\nSaved → {OUTPUT_PATH}')
print(f'File size: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB')

# ── Final policy window preview ───────────────────────────────────────────
print('\n--- Policy window (May 9–16, 2026) ---')
window = df.loc['2026-05-09':'2026-05-16',
                ['Gold_INR_PM', 'Gold_USD', 'rupees_per_dollar',
                 'parity_pre', 'domestic_premium', 'premium_pct', 'post_hike']]
print(window.to_string())

print('\n--- Pre/post summary ---')
policy_date = pd.Timestamp(POLICY_DATE)
pre  = df.loc[df.index < policy_date,  'domestic_premium'].dropna()
post = df.loc[df.index >= policy_date, 'domestic_premium'].dropna()
print(f'Pre-hike  n={len(pre):>4}  mean={pre.mean():>8,.0f}  std={pre.std():>7,.0f}  ' 
      f'min={pre.min():>9,.0f}  max={pre.max():>8,.0f}')
print(f'Post-hike n={len(post):>4}  mean={post.mean():>8,.0f}  std={post.std():>7,.0f}  '
      f'min={post.min():>9,.0f}  max={post.max():>8,.0f}')
print(f'Raw mean shift: +{post.mean() - pre.mean():,.0f} INR/10g')


=== STEP 1: Outlier diagnostic & fix ===
No zero Gold_INR_PM found.

Gold_USD spike dates (4): ['2025-10-21', '2026-01-30', '2026-02-03', '2026-03-19']
  → Nulled Gold_USD and all Gold_USD-derived columns.

Post-fix domestic_premium range: [-6,115, 13,635]
Non-NaN observations: 785

=== STEP 2: Dataset health checks ===
Shape (pre-clean): (1172, 25)  (2022-01-03 → 2026-07-01)
All dates unique : True
Weekend dates (1): ['2025-02-01']

--- NaN summary (final, before save) ---
  days_since_hike                1136  (96.9%)
  silver_premium                  387  (33.0%)
  premium_pct                     387  (33.0%)
  domestic_premium                387  (33.0%)
  Gold_INR_PM                     350  (29.9%)
  Silver_INR_PM                   350  (29.9%)
  MCX_Gold_close                  348  (29.7%)
  ibja_source                     348  (29.7%)
  delta_Gold_USD                   91  (7.8%)
  delta_Oil                        85  (7.3%)
  Nifty50                          64  (5.5%)
  GOLDB